# Do self-descriptions change over time *within* a family?

`self_descriptions.ipynb` (R-001) found that generators of the same family have distinguishable
forensic self-descriptions Φ, even with origin dataset fixed. This notebook asks the next
question: **does the distribution of Φ shift with release date inside a family**, as one would
expect if new, time-specific signals appear as a family evolves? Or are the differences between
generators unrelated to when they appeared?

Method (DECISIONS.md D-008, D-009):

* **Time bins.** Each family's release groups are split into equal-count bins by release date
  (3 bins primary; 2 and 4 as checks). One Gaussian mixture is fitted per bin over the feature
  space. Its components act as the bin's clusters, and their number is chosen by BIC.
* **The unit of evidence is a release, not an image.** Release date belongs to a generator, so the
  only temporal claim worth making is one that transfers to a generator the model has not seen.
  Every test holds out a whole **release group** (generators released within 7 days of each
  other, e.g. FLUX Dev + Schnell). Every null shuffles release dates **among groups**. Images
  inside one generator are not independent evidence about time.
* **T1, the headline.** Can an unseen release be placed in its time bin? Statistic: mean
  log-likelihood ratio of the true bin against the other bins (nats per image), averaged over
  release groups. Accuracy and mean bin error are also shown.
* **T2.** Do generators drift apart smoothly with time? Held-out divergence between
  (generator × origin) cells against their release-date gap.
* **T3, T4.** What the bin clusters are, and how finely each generator splits on its own.
* **Which variables.** Φ blocks (the mean-over-channels filter φ̄ vs the per-channel deviations;
  neighbourhood rings; offset parity) and the 10 extraction configs.

Features: robust scaling, then centring on **family × origin-dataset** means (so that origin
datasets, which are entangled with date, cannot carry the result by location alone), then the
pooled PCA of R-001, keeping g = 10 components.

**Status of this run: exploratory.** It is the local sample: 17 images per generator, 12–13
release groups per family. The feature set φ̄ was promoted after a pilot on these same data, so
its local p-values are optimistic. The confirmatory test is the pre-registered spec in D-008/D-009
on the server's stratified-36k.

Everything here is loaded. It is computed by
`python -m selfdesc.temporal --exp experiments/selfdesc-sample --perms 1000`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from selfdesc import temporal as T

ROOT = Path.cwd()
EXP = ROOT / "experiments" / "selfdesc-sample"
OUT = EXP / "temporal"

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
CONTEXT, NULLBAND, INK, MUTED = "#d9d8d4", "#e4e3df", "#0b0b0b", "#52514e"
SEQ = LinearSegmentedColormap.from_list("seq", ["#f4f3f0", BLUE, "#0b2a55"])
DIV = LinearSegmentedColormap.from_list("div", ["#a8401a", ORANGE, "#e9e8e4", BLUE, "#0b2a55"])
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.titlesize": 10, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
                     "legend.frameon": False})

meta = json.loads((OUT / "meta.json").read_text())
index = pd.read_parquet(EXP / "images.parquet")
t1 = pd.read_parquet(OUT / "t1_summary.parquet")
t1g = pd.read_parquet(OUT / "t1_generators.parquet")
t1n = pd.read_parquet(OUT / "t1_null.parquet")
power = pd.read_parquet(OUT / "t1_power.parquet")
t2 = pd.read_parquet(OUT / "t2_summary.parquet")
t2p = pd.read_parquet(OUT / "t2_pairs.parquet")
t3 = pd.read_parquet(OUT / "t3_components.parquet")
t4 = pd.read_parquet(OUT / "t4_generators.parquet")
size = pd.read_parquet(OUT / "t1_size.parquet")
FEATURE_NAME = {"full": "full Φ", "mean": "φ̄ (mean over channels)"}
print(f"config {meta['config']}, primary {meta['primary']}, {meta['perms']} permutations "
      f"(grid {meta['grid_perms']}, other configs {meta['config_perms']}), "
      f"release window {meta['release_window_days']} days, commit {str(meta['git_commit'])[:8]}, "
      f"{meta['seconds'] / 60:.0f} min")

## 1. The data: who is in which time bin, and from which dataset

One row per generator, in release order. Dots show which origin datasets its images come from
(area ∝ images). Shading marks the primary time bins; brackets mark release groups. Read across
the rows: **date and origin dataset are entangled.** Early diffusion models come from
DDMD/Artifact and FLUX only from SFHQ-T2I. This is why the features are centred on family ×
origin, and why per-pair checks restricted to one origin (T2) matter.

In [ ]:
def family_layout(fam):
    sub = index[index.generator_family == fam]
    raw = sub.drop_duplicates("generator").set_index("generator").release_date
    groups = T.release_groups(raw)
    dates = T.group_dates(raw, groups)
    bins = T.time_bins(dates, meta["primary"]["n_bins"], groups)
    order = raw.sort_values(kind="stable").index
    return sub, raw, groups, bins, order

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1, 1]})
for ax, fam in zip(axes, T.TIME_FAMILIES):
    sub, raw, groups, bins, order = family_layout(fam)
    origins = sub.groupby("origin_dataset").release_date.min().sort_values().index
    ct = pd.crosstab(sub.generator, sub.origin_dataset).loc[order, origins]
    for b in range(meta["primary"]["n_bins"]):
        rows = [i for i, g in enumerate(order) if bins[g] == b]
        ax.axhspan(min(rows) - 0.5, max(rows) + 0.5, color=NULLBAND if b % 2 else "white", zorder=0)
        ax.text(len(origins) - 0.4, np.mean(rows), f"bin {b}", va="center", fontsize=8, color=MUTED)
    yy, xx = np.nonzero(ct.values)
    ax.scatter(xx, yy, s=ct.values[yy, xx] * 14, color=BLUE, edgecolors="white", linewidths=0.5,
               zorder=3)
    for gid, members in groups.groupby(groups).groups.items():
        if len(members) > 1:
            rows = [list(order).index(m) for m in members]
            ax.plot([-0.75, -0.75], [min(rows) - 0.3, max(rows) + 0.3], color=INK, lw=1.5,
                    clip_on=False)
    ax.set_yticks(range(len(order)), [f"{g}  ({raw[g]:%Y-%m})" for g in order])
    ax.set_xticks(range(len(origins)), origins, rotation=55, ha="right")
    ax.set_xlim(-0.9, len(origins) + 0.3)
    ax.invert_yaxis()
    ax.set_title(f"{fam}: {len(order)} generators, {groups.nunique()} release groups")
plt.tight_layout(); plt.show()

## 2. T1: can an unseen release be placed in its time bin?

The four primary tests (Holm-corrected together), then the null distributions they are judged
against. `llr` is in nats per image: positive means held-out releases are better explained by
their own time bin's mixture than by the others. `acc` is bin accuracy at the release-group
level (chance = 1/3). `mae` is mean |bin error|.

In [ ]:
cols = ["family", "features", "n_generators", "n_groups", "n_images", "llr", "null_llr_mean",
        "null_llr_q95", "p_llr", "p_llr_holm", "acc", "chance_acc", "null_acc_mean", "p_acc",
        "mae", "null_mae_mean", "p_mae"]
primary = t1[t1.kind == "primary"].copy()
primary["features"] = primary.features.map(FEATURE_NAME)
primary[cols].round(3)

In [ ]:
specs = t1[t1.kind == "primary"]
fig, axes = plt.subplots(1, len(specs), figsize=(3.2 * len(specs), 2.8), sharey=True)
for ax, (_, s) in zip(axes, specs.iterrows()):
    null = t1n[t1n.spec == s.spec].llr
    ax.hist(null, bins=40, color=CONTEXT, edgecolor="white", linewidth=0.3)
    ax.axvline(s.llr, color=BLUE, lw=2)
    ax.text(s.llr, ax.get_ylim()[1] * 0.92, f" observed\n p={s.p_llr:.3f}", color=INK, fontsize=8,
            va="top")
    ax.set_title(f"{s.family}: {FEATURE_NAME[s.features]}")
    ax.set_xlabel("mean log-LR, true bin vs others (nats/image)")
axes[0].set_ylabel("permutations")
plt.tight_layout(); plt.show()

Per held-out generator: mean log-likelihood under each bin's mixture, centred per row (so only
the *relative* fit shows), with the true bin outlined. Rows are in release order. A temporal
signal would look like a diagonal band.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 5.4))
for ax, (_, s) in zip(axes, specs.iterrows()):
    g = t1g[t1g.spec == s.spec].sort_values("date", kind="stable")
    ll = g[[c for c in g.columns if c.startswith("ll_")]].dropna(axis=1).to_numpy()
    rel = ll - ll.mean(1, keepdims=True)
    lim = np.abs(rel).max()
    ax.imshow(rel, cmap=DIV, vmin=-lim, vmax=lim, aspect="auto")
    for i, tb in enumerate(g.true_bin):
        ax.add_patch(plt.Rectangle((tb - 0.5, i - 0.5), 1, 1, fill=False, ec=INK, lw=1.4))
    ax.set_yticks(range(len(g)), [f"{n} ({d:%Y})" for n, d in zip(g.generator, g.date)]
                  if ax is axes[0] or ax is axes[2] else [""] * len(g))
    ax.set_xticks(range(rel.shape[1]), [f"bin {b}" for b in range(rel.shape[1])])
    ax.set_title(f"{s.family}: {FEATURE_NAME[s.features]}", fontsize=9)
plt.tight_layout(); plt.show()

### 2a. What this test could have detected (planted drift on the real features)

A negative result only means something next to the test's power. Here a known drift is planted
into each family's **real** features and T1 is rerun. Every image of a release at relative time
t ∈ [0, 1] is shifted along a random direction by δ · t within-generator standard deviations.
Everything already in the data is kept: generator differences, origin effects, and any true
drift. Five random directions per δ, 99 permutations each. Each power value therefore rests on
5 runs and is coarse (steps of 0.2).

In [ ]:
pw = (power.groupby(["family", "features", "delta"])
      .agg(power=("p_llr", lambda p: float((p <= 0.05).mean())), llr=("llr", "mean"))
      .reset_index())
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, fam in zip(axes, T.TIME_FAMILIES):
    for feat, colour in (("full", ORANGE), ("mean", BLUE)):
        d = pw[(pw.family == fam) & (pw.features == feat)]
        ax.plot(d.delta, d.power, color=colour, lw=2, marker="o", ms=4,
                label=FEATURE_NAME[feat])
    ax.axhline(0.05, color=MUTED, ls=":", lw=0.8)
    ax.set_xscale("log", base=2); ax.set_xticks(sorted(pw.delta.unique()),
                                                [str(x) for x in sorted(pw.delta.unique())])
    ax.set_xlabel("planted drift δ (within-generator SDs over the date range)")
    ax.set_title(fam)
axes[0].set_ylabel("power at p ≤ 0.05")
axes[-1].legend(loc="lower right")
plt.tight_layout(); plt.show()
pw.pivot_table(index=["family", "features"], columns="delta", values="power")

### 2b. Is it just image resolution?

Generator output resolution rose over time, so a description computed on native pixels could
date a generator by its pixel scale alone. Two checks, same T1 machinery and 1000 permutations:

1. T1 on **log native size alone** (a 1-D feature). This asks whether size *could* explain T1.
   Its log-LR is not on the same scale as the others (a near-degenerate 1-D density); compare
   accuracy and p.
2. T1 on Φ / φ̄ after **regressing log size and its square out** of every component within
   the family. This asks whether anything is left after a mean-level size adjustment. It does
   not remove size effects on the *spread* of the features, which the per-bin mixtures can also
   use.

In [ ]:
gen_size = (index[index.generator_family.isin(T.TIME_FAMILIES)]
            .assign(short=lambda d: np.minimum(d.source_width, d.source_height))
            .groupby(["generator_family", "generator"])
            .agg(date=("release_date", "first"), median_short_side=("short", "median")))
for fam in T.TIME_FAMILIES:
    g = gen_size.loc[fam]
    rho = g.date.rank().corr(g.median_short_side.rank())
    print(f"{fam}: Spearman(release date, generator median native short side) = {rho:.2f}")
size[["family", "features", "size_r2", "llr", "null_llr_mean", "p_llr", "acc", "null_acc_mean",
      "p_acc"]].round(3)

## 3. T2: do generators drift apart smoothly with time?

Each point is a pair of (generator × origin) cells from different release groups. The y-axis is
their held-out divergence (how much worse each cell is explained by the other's mixture than by
its own). The x-axis is their release gap. Smooth drift predicts a rising trend. Same-origin
pairs (orange) are the comparison origin cannot explain, and they are few locally.

In [ ]:
display(t2.assign(features=t2.features.map(FEATURE_NAME))
          [["family", "features", "pairs", "n_pairs", "n_generators", "spearman", "null_mean",
            "null_q95", "p"]].round(3))
fams = [f for f in T.TIME_FAMILIES]
fig, axes = plt.subplots(2, len(fams), figsize=(5 * len(fams), 6.4), sharex="col")
for col, fam in enumerate(fams):
    for row, feat in enumerate(("full", "mean")):
        ax = axes[row, col]
        d = t2p[(t2p.family == fam) & (t2p.features == feat) & ~t2p.same_generator]
        other = d[~d.same_origin]
        same = d[d.same_origin]
        ax.scatter(other.gap_days / 365.25, other.divergence, s=12, color=CONTEXT,
                   edgecolors="none", label="different origin")
        ax.scatter(same.gap_days / 365.25, same.divergence, s=26, color=ORANGE,
                   edgecolors="white", linewidths=0.5, label="same origin")
        ax.axhline(0, color=MUTED, lw=0.6)
        ax.set_title(f"{fam}: {FEATURE_NAME[feat]}")
        if row == 1:
            ax.set_xlabel("release gap (years)")
        if col == 0:
            ax.set_ylabel("held-out divergence (nats/image)")
axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()

## 4. T3: what are each time bin's clusters?

Each bin's mixture is fitted on all its generators. Components are chosen by BIC, up to 8. The
table shows how many components each bin wants, and whether they line up with **generators**
or with **origin datasets** (ARI; 0 = unrelated, 1 = identical partition). The heatmaps show
each component's makeup (φ̄ features).

In [ ]:
from sklearn.metrics import adjusted_rand_score
rows = []
for (feat, fam, b), d in t3.groupby(["features", "family", "bin"]):
    rows.append(dict(features=FEATURE_NAME[feat], family=fam, bin=b, images=len(d),
                     generators=d.generator.nunique(), components=d.n_components.iloc[0],
                     ari_generator=adjusted_rand_score(d.generator, d.component)
                     if d.n_components.iloc[0] > 1 else np.nan,
                     ari_origin=adjusted_rand_score(d.origin_dataset, d.component)
                     if d.n_components.iloc[0] > 1 else np.nan))
display(pd.DataFrame(rows).round(2))

for fam in T.TIME_FAMILIES:
    d = t3[(t3.family == fam) & (t3.features == "mean")]
    bins = sorted(d.bin.unique())
    fig, axes = plt.subplots(1, len(bins), figsize=(4.2 * len(bins), 3.6))
    for ax, b in zip(np.atleast_1d(axes), bins):
        x = d[d.bin == b]
        ct = pd.crosstab(x.generator, x.component)
        ct = ct.loc[x.drop_duplicates("generator").sort_values("release_date").generator]
        ax.imshow(ct.values, cmap=SEQ, aspect="auto")
        for (i, j), v in np.ndenumerate(ct.values):
            if v:
                ax.text(j, i, v, ha="center", va="center", fontsize=7,
                        color="white" if v > ct.values.max() * 0.55 else INK)
        ax.set_yticks(range(len(ct)), ct.index)
        ax.set_xticks(range(ct.shape[1]), [f"c{c}" for c in ct.columns])
        ax.set_title(f"{fam}, bin {b}: {ct.shape[1]} component(s)")
    plt.tight_layout(); plt.show()

## 5. T4: how finely does each generator split on its own?

The BIC-chosen component count of each generator's own mixture. `max_testable` is the most
components BIC was allowed to consider (n / 5 images). Where a generator does split, the ARI
columns say whether the split follows its origin datasets, its source format, or its native
size. **At 17 images per generator this is close to unresolvable.** BIC on 17 points in 10
dimensions has little room to prefer more than one component. This section is the server
run's job, and is kept here so the server output drops in unchanged.

In [ ]:
d = t4[t4.features == "mean"].sort_values(["family", "release_date"])
display(d.groupby("family").n_components.value_counts().unstack(fill_value=0))
d.assign(release_date=d.release_date.dt.strftime("%Y-%m-%d"))[
    ["family", "generator", "release_date", "n_images", "max_testable", "n_components",
     "levels_origin_dataset", "ari_origin_dataset", "levels_source_format", "ari_source_format",
     "levels_size", "ari_size"]].round(2)

## 6. Which variables, and which settings, carry it?

T1 excess over the null (observed log-LR minus null mean) for every non-primary specification.
The sensitivity grid uses 200 permutations, other configs 100. Rows group by what was varied:

* **block**: a part of Φ alone. `deviations` = φ_k − φ̄ (the per-channel part); `mean ring r`
  = φ̄ restricted to offsets at Chebyshev distance r from the centre; `mean even/odd offsets` =
  φ̄ on the lattice a 2× upsampling stage acts on, vs the rest;
* **bins / granularity / centring / single origin**: the design choices;
* **config**: the other extraction settings (K, B, L, the authors' FRE).

Blocks were explored because the user asked which parts of the description matter. They are
**exploratory**: many cells, no multiplicity correction.

In [ ]:
grid = t1[t1.kind != "primary"].copy()
grid["excess"] = grid.llr - grid.null_llr_mean

def label(r):
    if r.kind == "block":
        return f"block: {r.features}"
    if r.kind == "config":
        return f"config: {r.config} [{r.features}]"
    if r.kind == "bins":
        return f"bins={r.n_bins} [{r.features}]"
    if r.kind == "granularity":
        return f"g={r.g} [{r.features}]"
    if r.kind == "centring":
        return f"centring={r.centring} [{r.features}]"
    return f"{r.kind}: {r.origin} [{r.features}]"

grid["row"] = grid.apply(label, axis=1)
H = grid.pivot_table(index="row", columns="family", values="excess", sort=False)
Pv = grid.pivot_table(index="row", columns="family", values="p_llr", sort=False)
fig, ax = plt.subplots(figsize=(5.2, 0.27 * len(H) + 1.2))
lim = np.nanmax(np.abs(H.values))
im = ax.imshow(H.values, cmap=DIV, vmin=-lim, vmax=lim, aspect="auto")
for (i, j), v in np.ndenumerate(H.values):
    if np.isfinite(v):
        p = Pv.values[i, j]
        ax.text(j, i, f"{v:+.2f}{'*' if p <= 0.05 else ''}", ha="center", va="center",
                fontsize=7, color="white" if abs(v) > 0.55 * lim else INK)
ax.set_yticks(range(len(H)), H.index, fontsize=7)
ax.set_xticks(range(H.shape[1]), H.columns)
ax.set_title("T1 log-LR excess over null (* p ≤ 0.05, uncorrected)")
fig.colorbar(im, ax=ax, shrink=0.5, pad=0.02).outline.set_visible(False)
plt.tight_layout(); plt.show()

print("share of specifications with p <= 0.05, by family and kind:")
print(grid.assign(sig=grid.p_llr <= 0.05).pivot_table(index="kind", columns="family", values="sig",
                                                      aggfunc="mean").round(2).to_string())

## 7. What the evidence says (local sample, 2026-10-11; DECISIONS.md R-002)

**Short answer: on this sample, a within-family time signal in self-descriptions is not
demonstrated.** For GANs there is evidence *against* a large drift. For diffusion there is a weak,
suggestive signal that cannot yet be separated from the rise in image resolution over time.

**1. Primary tests:** 4 tests, Holm-corrected; 1000 release-group permutations; n = 13 diffusion
/ 12 GAN release groups. None is significant.

| | log-LR (null mean) | p | Holm p | bin accuracy (chance 0.33) |
|---|---|---|---|---|
| diffusion, φ̄ | +0.29 (−0.14) | 0.096 | 0.38 | 0.46 |
| diffusion, full Φ | −0.02 (−0.12) | 0.28 | 0.85 | 0.36 |
| GAN, φ̄ | −0.22 (−0.18) | 0.53 | 0.99 | 0.21 |
| GAN, full Φ | −0.13 (−0.12) | 0.50 | 0.99 | 0.21 |

**2. What the tests could see (§2a).** A planted drift of **2 within-generator SDs** across a
family's date range is detected with power 0.8 (GAN) to 1.0 (diffusion). At ≤ 1 SD the power is
0–0.4. So the GAN null result rules out a drift that large along a typical direction. Smaller
drifts cannot be excluded at 17 images per generator.

**3. Diffusion: consistent in sign, weak in strength, entangled with resolution.**
* Across the sensitivity grid, the φ̄ excess over the null is positive in 12 of 14 settings
  (g, centring, bins = 4, K, B, L, the authors' FRE). It reaches p ≤ 0.05 only twice (K = 2;
  the authors' FRE), uncorrected. It vanishes with 2 bins (−0.03) and with B = 5. Every setting
  reuses the same 272 images and 13 releases, so this agreement is robustness to analysis
  choices, **not** independent replication.
* The per-generator heatmap (§2) shows the signal is mostly the early bin being recognised:
  DDPM, ADM, Palette, DDGAN/Glide/LDM, 2020–21. Those are exactly the 200–256 px generators.
  Across diffusion generators, native resolution tracks release date (Spearman 0.79), and
  **native size alone places releases in their bin better than φ̄** (accuracy 0.69, p = 0.018).
* Regressing size out (linear + quadratic) leaves the φ̄ signal where it was (+0.30, p = 0.085),
  and size explains about 1 % of feature variance. So the signal is not a mean-level size shift.
  A size effect on the spread of the features remains possible, and this sample cannot separate
  "newer generator" from "higher-resolution output".
* Within φ̄, rings 2 (+0.64) and 3+ (+0.48) and even offsets (+0.43) carry more than ring 1
  (+0.16). The per-channel deviations add nothing (+0.10). [I] Offsets of 2 px and more are the
  scale at which both upsampling design and output resolution act, so this does not discriminate
  between the two readings. Exploratory and uncorrected.

**4. GAN: nothing.** 1 of 36 sensitivity settings reaches p ≤ 0.05 (K = 2, full Φ), about what
chance gives. The excess is positive in only 14 of 36. Restricting to GANs from Artifact alone
(11 release groups, one origin) changes nothing.

**5. Smooth drift (T2).** In all four family × feature combinations, pairs further apart in time
are somewhat more divergent (ρ 0.08–0.27). Only diffusion φ̄ over all pairs reaches p ≈ 0.05, and
those pairs mix origin datasets. Same-origin pairs, the clean comparison, number 6 (diffusion) and
22 (GAN), with ρ ≤ 0.21 and p ≥ 0.19.

**6. Clusters.** Each bin's mixture wants 1–2 components. Where it wants 2, they follow neither
generators (ARI ≤ 0.08) nor origin datasets (≤ 0.02). Per generator, BIC picks one component for
13 of 16 diffusion and 10 of 13 GAN generators. At 17 images in 10 dimensions that is the expected
answer, not evidence that generators are internally homogeneous.

**How strong is the evidence, overall?** For "Φ shifts over time within GANs": moderately strong
evidence against a large shift, none for a small one. For "within diffusion": weak (primary
p ≈ 0.1, Holm 0.38), consistent in direction across analysis choices, and confounded with output
resolution. Nothing here supports dating an unseen generator from its self-description.

**What would settle it.** (a) The pre-registered spec (D-008/D-009) on the server's stratified-36k:
~30× the images, same frozen FREs, which also makes §5 resolvable. (b) A resolution-controlled
comparison: generators of equal native resolution only, or a crop policy that brings images to a
common pixel scale. The second departs from the no-resampling decision (D-006), so it would be a
separate, explicitly labelled arm.